# BEIR benchmark for the README table

Runs `experiments/beir_benchmark.py` end-to-end on Colab so the
numbers in the project README are reproducible and current. Default
model: `Stffens/bge-small-rrf-v2` (the published fine-tuned model).

Runtime on T4 for all five BEIR datasets: ~30-45 min. The slowest is
fiqa (57k docs).

In [ ]:
# Cell 1: Setup. Pull develop (has T1.1/T1.3 + latest bench tweaks).
!pip install -q 'sentence-transformers>=3' torch 'accelerate>=1.1.0' chromadb
!rm -rf /content/vstash
!git clone --branch develop https://github.com/stffns/vstash.git /content/vstash
%cd /content/vstash
!pip install -q -e .

In [ ]:
# Cell 2: Run the benchmark on the tuned model.
# Change --datasets to subset, or --model to evaluate a different HF model.
import os
from pathlib import Path

os.chdir("/content/vstash")
os.makedirs("experiments/data", exist_ok=True)

# The repo ships with an old results JSON (from 2026-04-10, v1 model,
# fiqa only). Delete it so Cell 3 cannot accidentally read stale data
# while this run is still in progress or if it fails mid-way.
stale = Path("experiments/results/beir_benchmark.json")
if stale.exists():
    stale.unlink()
    print(f"  cleared stale results file: {stale}")

!python -m experiments.beir_benchmark --no-chroma --model Stffens/bge-small-rrf-v2

In [ ]:
# Cell 3: Build a README-ready markdown table from the saved JSON.
# Run this ONLY after Cell 2 finishes (the bench can take 30-45 min
# across all five BEIR datasets on T4).
import json
from pathlib import Path

from experiments.beir_benchmark import BASELINES

results_path = Path("experiments/results/beir_benchmark.json")
assert results_path.exists(), "beir_benchmark.json not found. Cell 2 must complete first."
data = json.loads(results_path.read_text())
model = data["model"]
results = data["results"]

# Sanity check: the JSON must be from this run, not a stale file.
expected_model = "Stffens/bge-small-rrf-v2"
if model.lower() != expected_model.lower():
    raise RuntimeError(
        f"Results JSON is from a different model ({model!r}). "
        f"Re-run Cell 2 with --model {expected_model} and retry."
    )

print(f"Model: {model}")
print(f"Datasets: {len(results)}")
print()
print("| Dataset | Docs | vstash (tuned) | ColBERTv2 | BM25 | vs ColBERTv2 |")
print("|---------|:----:|:--------------:|:---------:|:----:|:------------:|")
for r in results:
    ds = r["dataset"]
    b = BASELINES.get(ds, {})
    v = r["vstash"]["ndcg_10"]
    col = b.get("ColBERTv2", 0)
    bm = b.get("BM25", 0)
    docs_label = f"{r['docs'] / 1000:.1f}K" if r["docs"] >= 1000 else str(r["docs"])
    delta = (v - col) / col * 100 if col else 0
    winner_v = "**" if v > col else ""
    winner_c = "**" if col > v else ""
    print(
        f"| {ds.capitalize()} | {docs_label} | "
        f"{winner_v}{v:.3f}{winner_v} | "
        f"{winner_c}{col:.3f}{winner_c} | "
        f"{bm:.3f} | "
        f"{'**' if delta > 0 else ''}{delta:+.1f}%{'**' if delta > 0 else ''} |"
    )